# TimeMachine QuantOptimizer: Quickstart & Strategy Simulation

Welcome to **TimeMachine QuantOptimizer**. This notebook demonstrates how to:
1. Load real and synthetic continuous futures market data.
2. Inspect the pre-loaded StockSharp quantitative strategies.
3. Execute an end-to-end backtest with tick-level commission and execution simulation.
4. Compute institutional risk metrics (Sharpe, Drawdown, Profit Factor).
5. Run a Monte Carlo resampling stress test.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from mnq_backtester.data.loader import DataLoader
from mnq_backtester.strategies.library import get_strategy_suite, PINE_STRATEGY_SUITE
from mnq_backtester.pinescript.parser import PineScriptRunner
from mnq_backtester.analytics.metrics import PerformanceMetrics
from mnq_backtester.monte_carlo.engine import MonteCarloEngine, ResamplingMethod

## 1. Inspect Available StockSharp Preset Strategies

In [ ]:
strategies = get_strategy_suite()
print(f"Loaded {len(strategies)} open-source quantitative strategies:")
for s in strategies[:10]:
    print(f" - [{s['category']}] {s['name']}")

## 2. Load Market Data & Reconstruct Intraday Bars

In [ ]:
# Load historical daily continuous futures
daily_candles = DataLoader.fetch_yahoo_daily(symbol="MNQ=F", days=120)
print(f"Total daily candles loaded: {len(daily_candles)}")

# Reconstruct high-fidelity 1-minute intraday bars for sample sessions
intraday_bars = []
for candle in daily_candles[-20:]:
    bars_1m = DataLoader.reconstruct_1m_bars(candle)
    intraday_bars.extend(bars_1m)

print(f"Generated {len(intraday_bars)} 1-minute microstructure bars across 20 sessions.")

## 3. Execute Strategy Backtest Simulation

In [ ]:
# Pick the first generic StockSharp strategy
selected_key = next(iter(PINE_STRATEGY_SUITE))
code = PINE_STRATEGY_SUITE[selected_key]["code"]

result, strategy_obj = PineScriptRunner.run_pine_code(code, intraday_bars)

print("Backtest Execution Complete:")
print(f"Strategy Name: {strategy_obj.name}")
print(f"Total Bars Evaluated: {result.total_bars_processed}")
print(f"Completed Trades: {len(result.trades)}")

## 4. Compute Performance & Institutional Risk Metrics

In [ ]:
stats = PerformanceMetrics.calculate(
    trades=result.trades,
    equity_curve=result.equity_curve,
    initial_capital=50000.0
)

print(stats.summary_table())

## 5. Monte Carlo Resampling Stress Testing

In [ ]:
mc = MonteCarloEngine(iterations=500, seed=42)
mc_report = mc.run(result.trades, initial_capital=50000.0, method=ResamplingMethod.IID_BOOTSTRAP)

print(mc_report.summary_table())